In [3]:
import requests, time, json, pathlib
import pandas as pd

WETTER = pathlib.Path("../data/raw/wetter"); WETTER.mkdir(parents=True, exist_ok=True)
gem = pd.read_csv("../data/processed/gemeinden.csv")
print(len(gem), "Gemeinden")          # muss 186 sein

def hat_sonne(datei):
    try:
        return "sunshine_duration" in json.loads(datei.read_text(encoding="utf-8")).get("daily", {})
    except Exception:
        return False

def fetch_wetter(row, start="2013-11-01", end="2026-04-30"):
    params = {
        "latitude": row.lat, "longitude": row.lon,
        "elevation": row.hoehe_m,
        "start_date": start, "end_date": end,
        "daily": "temperature_2m_mean,snowfall_sum,precipitation_sum,sunshine_duration",
        "models": "era5",
        "timezone": "Europe/Zurich",
    }
    for versuch in range(15):
        r = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params, timeout=90)
        if r.status_code != 429:
            r.raise_for_status()
            return r.json()
        try:
            grund = r.json().get("reason", "")
        except Exception:
            grund = ""
        print("429:", grund)
        if "aily" in grund:                                   # Tageslimit: aufhoeren
            raise RuntimeError("Tageslimit erreicht – morgen weiterlaufen lassen")
        warte = 65 if "inut" in grund else 600                # Minutenlimit: 1 Min, sonst 10 Min
        print(f"   warte {warte} s ...")
        time.sleep(warte)
    raise RuntimeError("Limit bleibt bestehen – spaeter erneut starten")

for row in gem.itertuples():
    ziel = WETTER / f"{row.bfs_nr}.json"
    if ziel.exists() and hat_sonne(ziel):
        continue
    ziel.write_text(json.dumps(fetch_wetter(row)), encoding="utf-8")
    print("ok", row.bfs_nr, row.name)
    time.sleep(3)

186 Gemeinden
ok 261 Zürich
ok 293 Wädenswil
ok 295 Horgen
ok 329 Langenthal
ok 351 Bern
429: Minutely API request limit exceeded. Please try again in one minute.
   warte 65 s ...
ok 355 Köniz
ok 371 Biel/Bienne
ok 404 Burgdorf
ok 561 Adelboden
ok 562 Aeschi bei Spiez
429: Minutely API request limit exceeded. Please try again in one minute.
   warte 65 s ...
ok 565 Kandersteg
ok 567 Reichenbach im Kandertal
ok 571 Beatenberg
ok 573 Brienz (BE)
ok 576 Grindelwald
429: Minutely API request limit exceeded. Please try again in one minute.
   warte 65 s ...
ok 581 Interlaken
ok 584 Lauterbrunnen
ok 587 Matten bei Interlaken
ok 593 Unterseen
ok 594 Wilderswil
429: Minutely API request limit exceeded. Please try again in one minute.
   warte 65 s ...
ok 700 Moutier (BE)
ok 762 Diemtigen
ok 768 Spiez
ok 783 Hasliberg
ok 784 Innertkirchen
429: Minutely API request limit exceeded. Please try again in one minute.
   warte 65 s ...
ok 785 Meiringen
ok 792 Lenk
ok 843 Saanen
ok 902 Langnau im Emme

RuntimeError: Tageslimit erreicht – morgen weiterlaufen lassen